# Capstone — Credit Risk Decision Engine

Run `!pip install optuna shap -q` and upload `loan_applications.csv` first, then run cells top to bottom.

## Cell 1 — Setup and imports

In [ ]:
# In Colab, run this first:
#     !pip install optuna shap -q
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, json, datetime, warnings

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_score, RandomizedSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (roc_auc_score, average_precision_score, confusion_matrix,
                             classification_report, roc_curve, precision_recall_curve)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RS = 42
print("Ready.")

## Cell 2 — Business framing

In [ ]:
# ---------------------------------------------------------------------------
# CLIENT   : a digital lending platform
# PROBLEM  : approve good borrowers fast, without absorbing unaffordable losses
# DECISION : for each application -> AUTO-APPROVE / MANUAL REVIEW / DECLINE
# ECONOMICS: a funded good loan earns  ~ Rs 14,000 over its life
#            a funded bad loan  loses  ~ Rs 78,000 (principal + collection cost)
#            a manual review costs     ~ Rs 900 in underwriter time
# CONSTRAINT: the underwriting team can review at most 15% of applications
# METRIC   : expected portfolio profit, NOT accuracy
# ---------------------------------------------------------------------------
PROFIT_GOOD_LOAN = 14000
LOSS_BAD_LOAN = -78000
COST_MANUAL_REVIEW = -900
REVIEW_CAPACITY = 0.15
print("A model that maximises AUC but ignores these numbers is not a solution.")

## Cell 3 — Load and inspect

In [ ]:
df = pd.read_csv("loan_applications.csv")
print("Raw shape:", df.shape)
print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("After dedupe:", df.shape)
print("\nDefault rate:", round(df["default"].mean(), 4))
df.head(3)

## Cell 4 — Data quality audit

In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(),
})
print(quality.to_string())

# months_since_last_delinq is missing BY DESIGN: it is NaN when the borrower has
# never been delinquent. That is information, not an error - so we flag it
# explicitly instead of letting the imputer silently invent a number.
df["has_prior_delinq"] = df["months_since_last_delinq"].notna().astype(int)
print("\nDefault rate by prior delinquency flag:")
print(df.groupby("has_prior_delinq")["default"].agg(["mean", "count"]).round(3))

## Cell 5 — EDA

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 8))

df["default"].value_counts().plot(kind="bar", ax=axes[0,0], color=["#2E7D57","#C0392B"])
axes[0,0].set_title("Target balance"); axes[0,0].tick_params(axis="x", rotation=0)

df.groupby("employment_type")["default"].mean().sort_values().plot(
    kind="barh", ax=axes[0,1], color="#1F4E79")
axes[0,1].set_title("Default rate by employment type")

bands = pd.cut(df["bureau_score"], [300,580,660,740,900])
df.groupby(bands, observed=True)["default"].mean().plot(
    kind="bar", ax=axes[0,2], color="#1F4E79")
axes[0,2].set_title("Default rate by bureau band"); axes[0,2].tick_params(axis="x", rotation=20)

sns.boxplot(data=df, x="default", y="emi_to_income", hue="default", legend=False,
            ax=axes[1,0], palette=["#2E7D57","#C0392B"])
axes[1,0].set_title("EMI-to-income by outcome"); axes[1,0].set_ylim(0, 1.2)

sns.histplot(data=df, x="income_monthly", hue="default", bins=50, log_scale=True,
             ax=axes[1,1], palette=["#2E7D57","#C0392B"])
axes[1,1].set_title("Income (log scale) by outcome")

numcols = df.select_dtypes(include=np.number).columns
sns.heatmap(df[numcols].corr(), cmap="RdBu_r", center=0, ax=axes[1,2], cbar=True)
axes[1,2].set_title("Correlation - note the dense bureau block")

plt.tight_layout(); plt.show()

## Cell 6 — Unsupervised: portfolio segmentation

In [ ]:
# Before predicting anything, understand WHO is in the book.
seg_feats = ["income_monthly","bureau_score","credit_utilization",
             "emi_to_income","employment_years","num_delinquencies_2y"]
seg = df[seg_feats].dropna()
Xseg = StandardScaler().fit_transform(seg)

inertia = [KMeans(k, n_init=10, random_state=RS).fit(Xseg).inertia_ for k in range(1, 9)]
plt.figure(figsize=(7,3.5))
plt.plot(range(1,9), inertia, "o-", color="#1F4E79")
plt.xlabel("K"); plt.ylabel("inertia"); plt.title("Elbow"); plt.tight_layout(); plt.show()

seg["segment"] = KMeans(4, n_init=10, random_state=RS).fit_predict(Xseg)
seg["default"] = df.loc[seg.index, "default"]
profile = seg.groupby("segment")[seg_feats].mean().round(2)
profile["n"] = seg["segment"].value_counts().sort_index()
profile["default_rate"] = seg.groupby("segment")["default"].mean().round(3)
print(profile.to_string())

# DBSCAN as an outlier detector, not a clusterer
db = DBSCAN(eps=1.2, min_samples=10).fit(Xseg)
seg["is_outlier"] = (db.labels_ == -1).astype(int)
print("\nOutliers flagged:", int(seg['is_outlier'].sum()),
      f"({seg['is_outlier'].mean():.1%})")
print("Default rate - outliers:", round(seg.loc[seg.is_outlier==1,"default"].mean(), 3),
      "| rest:", round(seg.loc[seg.is_outlier==0,"default"].mean(), 3))

## Cell 7 — Feature engineering

In [ ]:
def engineer(d):
    """Domain features. Every one has a credit-risk rationale."""
    d = d.copy()
    d["has_prior_delinq"]   = d["months_since_last_delinq"].notna().astype(int)
    d["debt_service_ratio"] = d["emi"] / d["income_monthly"].clip(lower=1)
    d["balance_per_line"]   = d["revolving_balance"] / d["num_credit_lines"].clip(lower=1)
    d["credit_age_years"]   = d["oldest_account_months"] / 12
    d["loan_to_income"]     = d["loan_amount"] / (d["income_monthly"].clip(lower=1) * 12)
    d["thin_file"]          = ((d["num_credit_lines"] <= 2) &
                               (d["oldest_account_months"] < 36)).astype(int)
    d["inquiry_intensity"]  = d["num_inquiries_6m"] / d["credit_age_years"].clip(lower=0.5)
    return d

feature_maker = FunctionTransformer(engineer)
print(engineer(df).filter(["debt_service_ratio","loan_to_income","thin_file",
                           "inquiry_intensity"]).describe().round(3).to_string())

## Cell 8 — Define X, y and split FIRST

In [ ]:
# customer identifier is dropped; months_since_last_delinq is replaced by the flag
# plus the raw column (the imputer handles it, the flag preserves the meaning).
X = df.drop(columns=["application_id", "default"])
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RS, stratify=y)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Default rate  train:", round(y_train.mean(),4), " test:", round(y_test.mean(),4))

## Cell 9 — Preprocessing with ColumnTransformer

In [ ]:
X_probe = engineer(X_train)                      # to discover post-engineering columns
numeric_features = X_probe.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in X_probe.columns if c not in numeric_features]
print("Numeric    :", len(numeric_features))
print("Categorical:", categorical_features)

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
])

def build(model):
    """Feature engineering -> preprocessing -> model, as ONE object."""
    return Pipeline([("features", feature_maker),
                     ("preprocessor", preprocessor),
                     ("model", model)])

## Cell 10 — Baselines (never skip this step)

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=RS)
baselines = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RS),
    "Random Forest":       RandomForestClassifier(n_estimators=300, random_state=RS, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=RS),
}
rows = []
for name, m in baselines.items():
    s = cross_val_score(build(m), X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    rows.append({"Model": name, "CV ROC-AUC": s.mean(), "std": s.std()})
    print(f"{name:<22} {s.mean():.4f} (+/- {s.std():.4f})")
baseline_table = pd.DataFrame(rows).round(4)

## Cell 11 — Hyperparameter tuning

In [ ]:
from scipy.stats import loguniform, randint
search_space = {
    "model__learning_rate":     loguniform(0.005, 0.3),
    "model__max_leaf_nodes":    randint(8, 64),
    "model__min_samples_leaf":  randint(10, 150),
    "model__l2_regularization": loguniform(1e-3, 10),
    "model__max_iter":          randint(100, 500),
}
search = RandomizedSearchCV(
    build(HistGradientBoostingClassifier(random_state=RS)),
    search_space, n_iter=25,
    cv=StratifiedKFold(3, shuffle=True, random_state=RS),
    scoring="roc_auc", random_state=RS, n_jobs=-1, refit=True)
search.fit(X_train, y_train)
print("Best CV ROC-AUC:", round(search.best_score_, 4))
for k, v in search.best_params_.items():
    print(f"  {k.replace('model__',''):<20} {round(v,4) if isinstance(v,float) else v}")
best_pipeline = search.best_estimator_

## Cell 12 — Honest evaluation on the untouched test set

In [ ]:
proba = best_pipeline.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)
print("Test ROC-AUC :", round(roc_auc_score(y_test, proba), 4))
print("Test PR-AUC  :", round(average_precision_score(y_test, proba), 4))
print("\n", classification_report(y_test, pred, target_names=["Repaid","Defaulted"]))
print("Confusion matrix [[TN FP][FN TP]]:\n", confusion_matrix(y_test, pred))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fpr, tpr, _ = roc_curve(y_test, proba)
axes[0].plot(fpr, tpr, color="#1F4E79", lw=2)
axes[0].plot([0,1],[0,1],"--",color="grey")
axes[0].set_xlabel("false positive rate"); axes[0].set_ylabel("true positive rate")
axes[0].set_title(f"ROC (AUC = {roc_auc_score(y_test, proba):.3f})")
pr, rc, _ = precision_recall_curve(y_test, proba)
axes[1].plot(rc, pr, color="#C0392B", lw=2)
axes[1].axhline(y_test.mean(), ls="--", color="grey")
axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision")
axes[1].set_title("Precision-Recall (dashed = random)")
plt.tight_layout(); plt.show()

## Cell 13 — Explainability: global

In [ ]:
perm = permutation_importance(best_pipeline, X_test, y_test, n_repeats=5,
                              random_state=RS, scoring="roc_auc", n_jobs=-1)
imp = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)
plt.figure(figsize=(9,5))
imp.head(12).sort_values().plot(kind="barh", color="#1F4E79")
plt.xlabel("drop in ROC-AUC when shuffled")
plt.title("What the model actually relies on")
plt.tight_layout(); plt.show()
print(imp.head(10).round(4))

## Cell 14 — Explainability: local, with SHAP

In [ ]:
import shap
pre_fitted = Pipeline(best_pipeline.steps[:-1])
model_only = best_pipeline.named_steps["model"]
feat_names = [n.split("__",1)[1] for n in
              best_pipeline.named_steps["preprocessor"].get_feature_names_out()]
Xt = pre_fitted.transform(X_test)

explainer = shap.TreeExplainer(model_only)
sv = np.array(explainer.shap_values(Xt[:300]))
if sv.ndim == 3:
    sv = sv[..., 1]

shap.summary_plot(sv, Xt[:300], feature_names=feat_names, show=False, max_display=12)
plt.tight_layout(); plt.show()

def explain_applicant(i):
    c = pd.Series(sv[i], index=feat_names).sort_values(key=abs, ascending=False)
    print(f"\nApplicant #{i} - predicted default probability {proba[i]:.1%}")
    print("Top factors increasing risk:")
    for f, v in c[c > 0].head(4).items():
        print(f"   + {f.replace('_',' ')}  ({v:+.2f} log-odds)")
    print("Top factors decreasing risk:")
    for f, v in c[c < 0].head(3).items():
        print(f"   - {f.replace('_',' ')}  ({v:+.2f} log-odds)")

explain_applicant(int(np.argmax(proba[:300])))
explain_applicant(int(np.argmin(proba[:300])))

## Cell 15 — Fairness audit

In [ ]:
def group_report(y_true, y_pred, groups):
    out = []
    for g in sorted(pd.Series(groups).unique()):
        m = (groups == g)
        tn, fp, fn, tp = confusion_matrix(y_true[m], y_pred[m], labels=[0,1]).ravel()
        out.append({"group": g, "n": int(m.sum()),
                    "actual_rate": y_true[m].mean(),
                    "selection_rate": y_pred[m].mean(),
                    "TPR": tp/(tp+fn) if (tp+fn) else np.nan,
                    "FPR": fp/(fp+tn) if (fp+tn) else np.nan})
    return pd.DataFrame(out).round(3)

for attr in ["gender", "region", "employment_type"]:
    rep = group_report(y_test.values, pred, X_test[attr].values)
    di = rep["selection_rate"].min() / rep["selection_rate"].max()
    print(f"\n--- {attr} ---")
    print(rep.to_string(index=False))
    print(f"Disparate impact ratio: {di:.3f}",
          "PASS (>=0.8)" if di >= 0.8 else "FAIL - investigate before deployment")

## Cell 16 — Turn probabilities into a three-way decision policy

In [ ]:
def simulate_policy(proba, y_true, decline_at, review_at):
    """DECLINE above decline_at, REVIEW between, AUTO-APPROVE below."""
    decision = np.where(proba >= decline_at, "DECLINE",
               np.where(proba >= review_at, "REVIEW", "APPROVE"))
    profit = 0.0
    # auto-approved: we fund them and live with the outcome
    m = decision == "APPROVE"
    profit += (y_true[m] == 0).sum() * PROFIT_GOOD_LOAN
    profit += (y_true[m] == 1).sum() * LOSS_BAD_LOAN
    # reviewed: pay the underwriter, and assume they catch 70% of the true bads
    m = decision == "REVIEW"
    profit += m.sum() * COST_MANUAL_REVIEW
    caught = (y_true[m] == 1).sum() * 0.70
    profit += (y_true[m] == 0).sum() * PROFIT_GOOD_LOAN
    profit += ((y_true[m] == 1).sum() - caught) * LOSS_BAD_LOAN
    # declined: no revenue, no loss
    return {"decline_at": decline_at, "review_at": review_at,
            "approve_%": (decision=="APPROVE").mean(),
            "review_%":  (decision=="REVIEW").mean(),
            "decline_%": (decision=="DECLINE").mean(),
            "profit_per_1000": profit / len(y_true) * 1000}

yv = y_test.values
results = []
for decline_at in [0.35, 0.45, 0.55, 0.65]:
    for review_at in [0.10, 0.15, 0.20, 0.25]:
        if review_at < decline_at:
            results.append(simulate_policy(proba, yv, decline_at, review_at))
policy = pd.DataFrame(results)
feasible = policy[policy["review_%"] <= REVIEW_CAPACITY]
print("Feasible policies (within the 15% manual-review capacity), best first:")
print(feasible.sort_values("profit_per_1000", ascending=False).head(8).round(3).to_string(index=False))

best_policy = feasible.sort_values("profit_per_1000", ascending=False).iloc[0]
print("\nCHOSEN POLICY:")
print(f"  Auto-approve below {best_policy['review_at']:.2f}")
print(f"  Manual review      {best_policy['review_at']:.2f} - {best_policy['decline_at']:.2f}")
print(f"  Decline above      {best_policy['decline_at']:.2f}")

## Cell 17 — Compare against the naive alternatives

In [ ]:
def flat_profit(strategy, proba, y_true):
    if strategy == "approve everyone":
        return ((y_true==0).sum()*PROFIT_GOOD_LOAN + (y_true==1).sum()*LOSS_BAD_LOAN)/len(y_true)*1000
    if strategy == "threshold 0.5, no review":
        d = (proba >= 0.5)
        return ((y_true[~d]==0).sum()*PROFIT_GOOD_LOAN +
                (y_true[~d]==1).sum()*LOSS_BAD_LOAN)/len(y_true)*1000
    return np.nan

print(f"{'approve everyone':<28} Rs {flat_profit('approve everyone', proba, yv):>12,.0f} per 1000 apps")
print(f"{'model, default 0.5 cutoff':<28} Rs {flat_profit('threshold 0.5, no review', proba, yv):>12,.0f} per 1000 apps")
print(f"{'tuned 3-way policy':<28} Rs {best_policy['profit_per_1000']:>12,.0f} per 1000 apps")

## Cell 18 — Package for deployment

In [ ]:
joblib.dump(best_pipeline, "credit_risk_model_v1.joblib")

model_card = {
    "model_name": "credit_risk_decision_engine",
    "version": "1.0.0",
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "algorithm": "HistGradientBoostingClassifier in a sklearn Pipeline",
    "hyperparameters": {k.replace("model__",""): (round(v,5) if isinstance(v,float) else v)
                        for k, v in search.best_params_.items()},
    "training_rows": int(len(X_train)),
    "test_roc_auc": round(float(roc_auc_score(y_test, proba)), 4),
    "raw_features_expected": list(X.columns),
    "decision_policy": {"auto_approve_below": float(best_policy["review_at"]),
                        "decline_above": float(best_policy["decline_at"])},
    "intended_use": "Route applications to auto-approve / manual review / decline.",
    "out_of_scope": "Not a final decision for protected-class-sensitive edge cases.",
    "known_limitations": [
        "Survivorship bias: we only observe outcomes for applicants who were funded.",
        "Region shows a disparate impact ratio below 0.8 - monitor and review.",
        "Gender correlates with income in this population.",
    ],
}
with open("model_card_v1.json", "w") as f:
    json.dump(model_card, f, indent=2)
print(json.dumps(model_card, indent=2))

## Cell 19 — The scoring service

In [ ]:
class CreditDecisionService:
    def __init__(self, model_path, card_path):
        self.pipe = joblib.load(model_path)
        self.card = json.load(open(card_path))
        self.expected = self.card["raw_features_expected"]
        self.policy = self.card["decision_policy"]

    def decide(self, payload: dict) -> dict:
        missing = [c for c in self.expected if c not in payload]
        if missing:
            raise ValueError(f"Missing fields: {missing[:5]}")
        row = pd.DataFrame([payload])[self.expected]
        p = float(self.pipe.predict_proba(row)[0, 1])
        if p >= self.policy["decline_above"]:
            action = "DECLINE"
        elif p >= self.policy["auto_approve_below"]:
            action = "MANUAL_REVIEW"
        else:
            action = "AUTO_APPROVE"
        return {"default_probability": round(p, 4), "action": action,
                "model_version": self.card["version"],
                "scored_at": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")}

svc = CreditDecisionService("credit_risk_model_v1.joblib", "model_card_v1.json")
for i in [int(np.argmax(proba[:300])), int(np.argmin(proba[:300])), 5]:
    print(f"applicant #{i}:", json.dumps(svc.decide(X_test.iloc[i].to_dict())))

## Cell 20 — Monitoring plan

In [ ]:
def psi(expected, actual, buckets=10):
    expected, actual = pd.Series(expected).dropna(), pd.Series(actual).dropna()
    edges = np.unique(np.quantile(expected, np.linspace(0, 1, buckets + 1)))
    e = np.clip(np.histogram(expected, bins=edges)[0] / len(expected), 1e-4, None)
    a = np.clip(np.histogram(actual,   bins=edges)[0] / len(actual),   1e-4, None)
    return float(np.sum((a - e) * np.log(a / e)))

rng = np.random.default_rng(7)
next_quarter = X_test.copy()
next_quarter["income_monthly"] *= rng.uniform(0.55, 0.85, len(next_quarter))

print("PSI vs training distribution:")
for col in ["income_monthly","bureau_score","emi_to_income","credit_utilization","loan_amount"]:
    v = psi(X_train[col], next_quarter[col])
    print(f"  {col:<20} {v:6.3f}  " +
          ("INVESTIGATE" if v > 0.25 else "watch" if v > 0.10 else "stable"))

print(f"\nMean predicted risk: baseline {proba.mean():.3f} -> "
      f"incoming {best_pipeline.predict_proba(next_quarter)[:,1].mean():.3f}")
print("""
MONITORING PLAN
  Daily   : prediction volume, latency, error rate, share of each action
  Weekly  : PSI on the top 10 features; score distribution vs baseline
  Monthly : group metrics by gender/region; override rate from underwriters
  Quarterly: realised default rate on matured loans -> true AUC
  RETRAIN IF: PSI > 0.25 on any top-5 feature, OR live AUC drops below 0.80,
              OR the underwriter override rate exceeds 20%
""")

## Cell 21 — Final recommendation

In [ ]:
print(f"""
RECOMMENDATION TO THE CREDIT COMMITTEE
======================================
1. MODEL. Gradient boosting inside a Pipeline, tuned with 25 randomised trials.
   Test ROC-AUC {roc_auc_score(y_test, proba):.3f}. The Pipeline is the deliverable:
   it accepts a raw application dict and handles imputation, scaling and encoding
   internally, so training and serving cannot drift apart.

2. POLICY. Auto-approve below {best_policy['review_at']:.2f}, manual review to
   {best_policy['decline_at']:.2f}, decline above. This sends {best_policy['review_%']:.1%}
   of applications to underwriters, inside the 15% capacity limit, and returns
   Rs {best_policy['profit_per_1000']:,.0f} per 1,000 applications against
   Rs {flat_profit('approve everyone', proba, yv):,.0f} for approving everyone.

3. FAIRNESS. Gender passes the four-fifths rule. REGION DOES NOT. Do not deploy
   region as a feature until legal has reviewed it; re-run the audit afterwards.

4. EXPLAINABILITY. SHAP produces per-applicant reasons, which is what adverse
   action notices legally require. Feature importance alone does not satisfy this.

5. RISKS. Survivorship bias is the big one - we only see outcomes for loans that
   were funded under the previous policy, so the model has never observed how
   rejected applicants would have behaved. Budget for a small random-approval
   holdout to collect that data honestly.
""")